### How different $\sigma$ structures affect EFD: A Glimpse

In [1]:
include("src/domain.jl");
include("src/lvmodel.jl");
include("src/trophic.jl");
include("src/core.jl");

In [2]:
# Omega1.0 and normalized volume are different
# since the geometry is different (linear vs quadratic)
# but the normalized volume is effectively the same now

function this_ep(σ::Matrix{Float64}, m::Vector{Float64}, S::Float64)
    d = fill(0.0, 3); N⁰ = fill(0.0, 3); K = 3
    Λ = inv(σ); Q = (Λ + Λ')/2; c = -Λ * d
    return EnergyConstrProb(σ,Λ,Q,c,m,d,N⁰,K,S,:Individual)
end

function supply_volume(m::Vector{Float64}, S::Float64, K::Int=3)
    return((K*S)^K / (factorial(K) * prod(m)))
end

using RCall
R"library(feasibilityR)"

function feasby(matA)
    @rput matA  # Send the Julia matrix to R
    Ω = R"feasibility_community(matA)"  # Call the R function
    return 8*rcopy(Ω)  # Convert the result to a native Julia type
end

σ = abs.(randn(3,3))
m = fill(1.0,3)
p = this_ep(σ, m, 1.0)
feasby(-σ), vol_by_constr(p)/supply_volume(m, 1.0)

(0.12785837737178443, 0.09563242974566868)

For each type of networks, generate 50 samples, generate their $V(S)$ curves

In [ ]:
using DataFrames
df = DataFrame(rep_id=Int[], type=String[], S_ind=Float64[], vol=Float64[])
S_levels = 0.5:0.025:10.0;

for type in ["hierarchy", "omnivory", "exploitative", "competitive"]
    for i in 1:50
        σ = generate_trophic(type)
        for S_ind in S_levels
            p = create_ep_trophic(σ, S_ind, :Individual)
            vol = vol_by_constr(p)
            push!(df, (i, type, S_ind, vol))
        end
        @info "Iter for i:$i, type:$type finished"
    end
end

In [ ]:
using CSV, DataFrames
CSV.write("analysis/trophic_data.csv", df)

In [ ]:
using DataFrames, Statistics, StatsBase, Plots, StatsPlots

grouped_data = combine(groupby(df, [:S_ind, :type]), :vol => mean => :mean_vol)

@df grouped_data plot(:S_ind, :mean_vol, group=:type, lw=2, 
                      xlabel="S_ind", ylabel="Mean Volume", 
                      title="Mean Volume vs. S_ind", legend=:topright)

Vanilla case: Purely scalable domain, relationship with Omega

In [16]:
using DataFrames
df_plain = DataFrame(rep_id=Int[], type=String[], S_ind=Float64[], vol=Float64[], omega=Float64[])
S_levels = 0.5:0.025:10.0;

for i in 1:50
    σ = abs.(randn(3,3))
    omega = feasby(σ)
    for S_ind in S_levels
        p = this_ep(σ, m, S_ind)
        vol = vol_by_constr(p)
        push!(df_plain, (i, "Plain", S_ind, vol, omega))
    end
    @info "Iter for i:$i finished"
end

┌ Info: Iter for i:1 finished
└ @ Main /Users/longcy/Documents/Codefield/energy-constrain/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sZmlsZQ==.jl:13
┌ Info: Iter for i:2 finished
└ @ Main /Users/longcy/Documents/Codefield/energy-constrain/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sZmlsZQ==.jl:13
┌ Info: Iter for i:3 finished
└ @ Main /Users/longcy/Documents/Codefield/energy-constrain/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sZmlsZQ==.jl:13
┌ Info: Iter for i:4 finished
└ @ Main /Users/longcy/Documents/Codefield/energy-constrain/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sZmlsZQ==.jl:13
┌ Info: Iter for i:5 finished
└ @ Main /Users/longcy/Documents/Codefield/energy-constrain/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sZmlsZQ==.jl:13
┌ Info: Iter for i:6 finished
└ @ Main /Users/longcy/Documents/Codefield/energy-constrain/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X12sZmlsZQ==.jl:13
┌ Info: Iter for i:7 finished
└ @ Main /Users/

In [18]:
using CSV, DataFrames
CSV.write("analysis/plain_data.csv", df_plain)

"analysis/plain_data.csv"